In [13]:
import torch
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Lambda

torch.random.manual_seed(42)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
learning_rate = 1e-3
batch_size = 64
epochs = 10

full_train_dataset = MNIST(root='./data', train=True, download=True, transform=ToTensor())
test_dataset = MNIST(root='./data', train=False, download=True, transform=ToTensor())

device

device(type='mps')

In [14]:
from torch.utils.data import Dataset, DataLoader, random_split

lenght = len(full_train_dataset)
train_dataset, val_dataset = random_split(full_train_dataset, [lenght - 10000, 10000])

full_train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [15]:
from torch import nn, optim

def conv_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 8),
            conv_block(8, 16),
            conv_block(16, 32),
        )
        self.head = nn.Sequential(
            self.features,
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(32, 10),
        )
    
    def forward(self, x):
        return self.head(x)

model = NeuralNetwork().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
sheduler = lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, "min", patience=10, factor=0.1)

In [16]:
from trainer import Trainer

trainer = Trainer(
    model,
    loss_fn,
    optimizer,
    device,
    scheduler=sheduler,
    epoch_amount=epochs,
)
trainer.fit(train_loader, val_loader)

Эпоха: 0 Loss_train: 1.1267073948669921, 0:00:05.178101 сек
Loss_val: 0.3594506360163355

Эпоха: 1 Loss_train: 0.5420709263790598, 0:00:05.301522 сек
Loss_val: 0.20809931787335947

Эпоха: 2 Loss_train: 0.4319975561536182, 0:00:05.412345 сек
Loss_val: 0.18723066260290752

Эпоха: 3 Loss_train: 0.3707792099631961, 0:00:05.464263 сек
Loss_val: 0.13762070997885079

Эпоха: 4 Loss_train: 0.32981762000361975, 0:00:05.445719 сек
Loss_val: 0.12523725080736883

Эпоха: 5 Loss_train: 0.3028575532386065, 0:00:05.292248 сек
Loss_val: 0.10928309465622067

Эпоха: 6 Loss_train: 0.279893649346612, 0:00:05.425716 сек
Loss_val: 0.09883550779336388

Эпоха: 7 Loss_train: 0.266490708226743, 0:00:05.376695 сек
Loss_val: 0.1006599006355758

Эпоха: 8 Loss_train: 0.2519413505768989, 0:00:05.354286 сек
Loss_val: 0.08440080216854431

Эпоха: 9 Loss_train: 0.23810423186520482, 0:00:05.535325 сек
Loss_val: 0.08529101524525767



In [17]:
def accuracy(logits, target):
    logits = torch.as_tensor(logits)
    target = torch.as_tensor(target)
    predicted_labels = logits.argmax(dim=1)
    return (predicted_labels == target).float().mean().item()

accuracy(trainer.predict(test_loader), test_loader.dataset.targets)

0.9803000092506409